In [ ]:
import os
import sys

# Clone the SoccerTrack repository
if not os.path.isdir("/content/SoccerTrack"):
    print("Cloning SoccerTrack repository...")
    os.system("git clone https://github.com/JulezKlein/SoccerTrack.git /content/SoccerTrack")
    print("Repository cloned")
else:
    print("SoccerTrack repository already exists")

# Change to repo directory
os.chdir("/content/SoccerTrack")
sys.path.insert(0, "/content/SoccerTrack")

In [ ]:
# Colab / notebook configuration
CONFIG_PATH = "configs/train_config.yaml"
DATASET = "soccertrack"   # "football" or "soccertrack"
VIEW_TYPE = "wide_view" # only used for soccertrack
MODEL = "yolov8s"     # e.g. yolov8s, yolo26n, rtdetr-l
EPOCHS = 50
IMG_SIZE = 640
BATCH = 16
EXPORT_ACTIVE = True
EXPORT_FORMAT = "both"  # coreml, onnx, both

# Mount Google Drive (for Colab data access)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = "/content/drive/MyDrive/datasets"
    print("Google Drive mounted")
except Exception:
    DATA_DIR = "./data"
    print("Google Colab not detected - running in local environment")

print(f"Using data dir: {DATA_DIR}")

In [ ]:
import torch

print("\n" + "="*60)
print("Environment Setup")
print("="*60)
print(f"PyTorch version: {torch.__version__}")
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device available: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA available: True")
    os.system("nvidia-smi")
else:
    print("CUDA available: False")

In [ ]:
# Install dependencies
print("\n" + "="*60)
print("Installing Dependencies")
print("="*60)

import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip", "-q"], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install",
     "ultralytics", "pandas", "pillow", "tqdm", "opencv-python", "matplotlib", "omegaconf", "-q"],
    check=True
)
print("Dependencies installed")

In [ ]:
# Import config-based training functions
from omegaconf import OmegaConf
from train.train_ultralytics_local import load_config, unzip_dataset, prepare_dataset, start_training, CONFIG

print("\n" + "="*60)
print("Building Colab Config")
print("="*60)

cfg = OmegaConf.load(CONFIG_PATH)
cfg.dataset.type = DATASET
cfg.dataset.view_type = VIEW_TYPE
cfg.training.model = MODEL
cfg.training.epochs = EPOCHS
cfg.training.img_size = IMG_SIZE
cfg.training.batch = BATCH

cfg.paths.data_dir = DATA_DIR
cfg.paths.soccertrack.dataset_zip = f"{DATA_DIR}/soccertrack.zip"
cfg.paths.soccertrack.dataset_dir = f"{DATA_DIR}/soccertrack"
cfg.paths.soccertrack.output_dir = f"{DATA_DIR}/output_yolo_soccertrack_{VIEW_TYPE}"
cfg.paths.football.dataset_dir = f"{DATA_DIR}/combined_dataset"
cfg.paths.football.output_dir = f"{DATA_DIR}/output_{MODEL}_football"

cfg.export.active = EXPORT_ACTIVE
cfg.export.format = EXPORT_FORMAT

COLAB_CONFIG_PATH = "configs/train_config_colab.yaml"
OmegaConf.save(cfg, COLAB_CONFIG_PATH)
config = load_config(COLAB_CONFIG_PATH)

In [ ]:
# Prepare dataset (only if needed for SoccerTrack)
print("\n" + "="*60)
print("Dataset Preparation")
print("="*60)

if config["requires_preparation"]:
    print(f"Preparing {config['dataset_type']} dataset for training...")
    unzip_dataset()
    prepare_dataset()
    print("Dataset preparation complete")
else:
    print(f"{config['dataset_type']} dataset is already in expected format")
    print("No preparation needed - data should be available at:")
    print(f"{config['dataset_dir']}")

In [ ]:
# Optional visualization
print("\n" + "="*60)
print("Dataset Visualization")
print("="*60)
print("Visualization helper was removed from the training script.")
print("Use utils/visualization.py if you want custom previews.")

In [ ]:
# Validate dataset path before training
from pathlib import Path

print("\n" + "="*60)
print("Validating Dataset Path")
print("="*60)

yaml_path = Path(config["yaml_path"])
if yaml_path.exists():
    print(f"Dataset YAML found: {yaml_path}")
else:
    print(f"Dataset YAML not found yet: {yaml_path}")
    print("If this is SoccerTrack, run dataset preparation first.")

In [ ]:
# Print training configuration
print("\n" + "="*60)
print("Training Configuration")
print("="*60)
print(f"Dataset: {config['dataset_type']}")
if config["dataset_type"] == "soccertrack":
    print(f"View Type: {config['view_type']}")
print(f"Model: {config['training']['model']}")
print(f"Epochs: {config['training']['epochs']}")
print(f"Image Size: {config['training']['img_size']}")
print(f"Batch: {config['training']['batch']}")
print(f"Output Dir: {config['output_dir']}")
print(f"YAML Path: {config['yaml_path']}")
print("="*60)

In [ ]:
# Start training
print("\n" + "="*60)
print(f"Starting {config['training']['model'].upper()} Training")
print("="*60)

try:
    start_training()
    print("\nTraining completed successfully!")
    print(f"Results saved to: {config['output_dir']}/{config['training']['model']}")
except Exception as e:
    print(f"\nTraining failed: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# Optional: Check training results
from pathlib import Path

model_name = config["training"]["model"]
output_dir = Path(config["output_dir"]) / model_name

if output_dir.exists():
    print("\n" + "="*60)
    print("Training Results")
    print("="*60)

    result_dirs = list(output_dir.glob("train*"))
    if result_dirs:
        latest_run = max(result_dirs, key=lambda p: p.stat().st_mtime)
        print(f"Latest training run: {latest_run.name}")

        weights_dir = latest_run / "weights"
        if weights_dir.exists():
            print("\nWeights:")
            for weight_file in weights_dir.glob("*.pt"):
                print(f"  - {weight_file.name}")

        results_csv = latest_run / "results.csv"
        if results_csv.exists():
            print(f"\nResults logged to: {results_csv}")
    else:
        print("No completed training runs found yet")
else:
    print(f"Output directory not found yet: {output_dir}")

In [ ]:
# Optional: Download trained model from Colab
try:
    from google.colab import files
    from pathlib import Path

    model_name = config["training"]["model"]
    model_dir = Path(config["output_dir"]) / model_name
    run_dirs = sorted(model_dir.glob("train*"), key=lambda p: p.stat().st_mtime)

    if run_dirs:
        weights_path = run_dirs[-1] / "weights" / "best.pt"
        if weights_path.exists():
            print(f"\nDownloading trained model: {weights_path.name}")
            files.download(str(weights_path))
            print("Model download started")
        else:
            print("No best.pt found in latest training run")
    else:
        print("No training run found to download from")
except ImportError:
    print("(Google Colab environment not detected - skipping download)")

In [ ]:
# ============================================================
# EXPORT CONFIGURATION
# ============================================================
from pathlib import Path
from utils.export_yolo import run_export

export_cfg = cfg.get("export", {})
EXPORT_FORMAT = str(export_cfg.get("format", "both"))
CONF_THRESH = float(export_cfg.get("conf_thresh", 0.5))
IOU_THRESH = float(export_cfg.get("iou_thresh", 0.7))

model_name = config["training"]["model"]
model_dir = Path(config["output_dir"]) / model_name
run_dirs = sorted(model_dir.glob("train*"), key=lambda p: p.stat().st_mtime)

if run_dirs:
    latest_run = run_dirs[-1]
    WEIGHTS = str(latest_run / "weights" / "best.pt")
else:
    WEIGHTS = str(model_dir / "weights" / "best.pt")

print(f"Using weights path: {WEIGHTS}")
print("\nExport Configuration:")
print(f"  Active: {bool(export_cfg.get('active', True))}")
print(f"  Format: {EXPORT_FORMAT}")
print(f"  Conf Threshold: {CONF_THRESH}")
print(f"  IoU Threshold: {IOU_THRESH}")

In [ ]:
# ============================================================
# EXPORT HELPERS
# ============================================================
from pathlib import Path

def resolve_best_weights():
    model_name = config["training"]["model"]
    model_dir = Path(config["output_dir"]) / model_name
    run_dirs = sorted(model_dir.glob("train*"), key=lambda p: p.stat().st_mtime)
    if run_dirs:
        candidate = run_dirs[-1] / "weights" / "best.pt"
        if candidate.exists():
            return str(candidate)

    fallback = model_dir / "weights" / "best.pt"
    return str(fallback)

WEIGHTS = resolve_best_weights()
print(f"Resolved weights: {WEIGHTS}")

In [ ]:
# ============================================================
# RUN EXPORTS
# ============================================================
from pathlib import Path

print("\n" + "="*60)
print("Model Export")
print("="*60)

if Path(WEIGHTS).exists() and bool(export_cfg.get("active", True)):
    try:
        run_export(
            weights=WEIGHTS,
            img_size=config["training"]["img_size"],
            conf_thresh=CONF_THRESH,
            iou_thresh=IOU_THRESH,
            export_format=EXPORT_FORMAT,
            data_yaml=config["yaml_path"]
        )
        print("\nExport pipeline complete!")
    except Exception as e:
        print(f"Export failed: {e}")
else:
    print(f"Skipping export. Weights not found or export inactive: {WEIGHTS}")

In [ ]:
# ============================================================
# DOWNLOAD EXPORTED MODELS (Colab only)
# ============================================================
import os
from pathlib import Path

try:
    from google.colab import files

    print("\n" + "="*60)
    print("Downloading Exported Models")
    print("="*60)

    files_to_download = []
    weights_path = Path(WEIGHTS)

    if EXPORT_FORMAT in ["coreml", "both"]:
        mlpackage_path = weights_path.with_suffix(".mlpackage")
        if mlpackage_path.exists():
            print(f"Found Core ML model: {mlpackage_path.name}")
            files_to_download.append(str(mlpackage_path))

    if EXPORT_FORMAT in ["onnx", "both"]:
        onnx_path = weights_path.with_suffix(".onnx")
        if onnx_path.exists():
            print(f"Found ONNX model: {onnx_path.name}")
            files_to_download.append(str(onnx_path))

    if weights_path.exists():
        print(f"Found PyTorch weights: {weights_path.name}")
        files_to_download.append(str(weights_path))

    if files_to_download:
        print(f"\nDownloading {len(files_to_download)} file(s)...")
        for file_path in files_to_download:
            print(f"  → {os.path.basename(file_path)}")
            files.download(file_path)
        print("Download complete!")
    else:
        print("No exported models found to download")

except ImportError:
    print("\n" + "="*60)
    print("Export Summary")
    print("="*60)
    print("Google Colab environment not detected")
    print("(Skipping automatic download)")
    print("\nExported model location:")
    print(str(Path(WEIGHTS).parent))